# ADS Bibliometric Analysis Pipeline

This notebook is the single entry point for the `ads_bib` package.
It executes all steps sequentially, with configuration cells between steps
so that downstream parameters can be set based on upstream results.

**Pipeline Steps:**
1. Search & Export â€” Query ADS API, resolve bibcodes to metadata
2. Translation â€” Detect languages, translate non-English titles/abstracts
3. Tokenization â€” Create full_text, tokenize with spaCy
4. AND â€” Author Name Disambiguation (placeholder)
5. Topic Modeling & Curation â€” BERTopic + datamapplot + cluster removal
6. Citation Networks â€” Direct, Co-Citation, Bibliographic Coupling, Author Co-Citation

---
## Setup

In [1]:
import os
from pathlib import Path

# Initialize paths (relative to this notebook's location)
from ads_bib.config import init_paths, load_env
from ads_bib.run_manager import RunManager
from ads_bib._utils.costs import CostTracker

paths = init_paths()  # uses CWD = notebook directory
load_env()

# Cost tracker for all API calls (OpenRouter, etc.)
tracker = CostTracker()

# --- INITIALIZE RUN ---
# This tracks all parameters and saves outputs to a unique folder
run = RunManager(run_name='ADS_Curation_Run')

In [2]:
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)
formatter = logging.Formatter("%(message)s")

if root_logger.handlers:
    for handler in root_logger.handlers:
        handler.setLevel(logging.INFO)
        handler.setFormatter(formatter)
else:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    handler.setFormatter(formatter)
    root_logger.addHandler(handler)

logger = logging.getLogger("pipeline")

---
## Global Run Control

Set run-level switches here. Phase-specific parameters are configured in each phase section below.


In [3]:
import random
import numpy as np

# ── CONFIGURE ─────────────────────────────────────────
# 0 = Run everything (Search to End)
# 4 = Load processed data (Skip Search, Translate, Tokenize) -> Start at Topic Modeling
START_AT_PHASE = 0
FORCE_REFRESH = True

# Deterministic seed for notebook-side randomness (sampling + reduction params)
RANDOM_SEED = 42

# Shared OpenRouter pricing mode (used in translation, embeddings, topic labeling)
OPENROUTER_COST_MODE = "hybrid"  # 'hybrid', 'strict', 'fast'
# ──────────────────────────────────────────────────────

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

---
# Phase 1: Search & Export

Query the NASA ADS API for publications matching a search string,
then resolve all bibcodes (publications + references) into full metadata.

### 1.1 Search Configuration

Set query, years, and retrieval limits for your research question.
These values define the corpus scope for all downstream steps.


In [4]:
# --- SEARCH CONFIGURATION ---
# Modify the query string for your research question.
# See: https://ui.adsabs.harvard.edu/help/search/search-syntax

ADS_TOKEN = os.getenv("ADS_API_KEY")

Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"
Set_B = f"citations({Set_A}) AND year:1915-2000"
Set_C = f"citations(citations({Set_A})) AND year:1915-2000"
Set_D = f"({Set_A}) OR ({Set_B}) OR ({Set_C})"
Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

gravity_relativity_terms = [
    '(general AND relativi*)', 'gravit*', '(allgemein* AND relativi*)',
    '"relativité générale"', '"teoria della relatività"',
    '"Gravité quantique"', '"Gravità quantistica"',
    '(einheitlich* AND feld*)', 'Quantengravitation', '"champ unifié"',
    '(unified AND field*)', '"quantum gravity"', 'cosmo*', 'Kosmo*',
]
Set_E = f"abs:({' OR '.join(gravity_relativity_terms)}) AND year:1915-2000"

# Example quick focus query
QUERY = 'author:"Treder, H*"'
# Alternative broader query:
# QUERY = f"({Set_D}) OR ({Set_T}) OR ({Set_E})"

In [5]:
# # ...existing code...
# # --- SEARCH CONFIGURATION ---
# # Modify the query string for your research question.
# # See: https://ui.adsabs.harvard.edu/help/search/search-syntax

# ADS_TOKEN = os.getenv("ADS_API_KEY")

# # 1. Historischer Seed: Einsteins Publikationen (1911-1955)
# Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"

# # 2. Direkte Rezeption: 1-Hop Zitationen ab 1915
# Set_B = f"citations({Set_A}) AND year:1915-2000"

# # 3. Strukturelle Anker-Begriffe (Mehrsprachig: EN, DE, FR, IT)
# generic_anchors = [
#     'relativ*', 'einstein*', 
#     'spacetime', '"space-time"', 'raumzeit', '"espace-temps"', 'spaziotempo', '"spazio-tempo"',
#     'tensor*', 'tenseur*', 
#     'metric*', 'metrik*', 'metriqu*', 
#     'curvature', 'krümmung', 'kruemmung', 'courbure', 'curvatura',
#     'geodesic*', 'geodät*', 'geodaet*', 'geodesiqu*', 'geodetic*'
# ]

# # 4. Indirekte Rezeption: 2-Hop Zitationen ab 1915 (Gefiltert durch Anker)
# Set_C = f"citations(citations({Set_A})) AND year:1915-2000 AND abs:({' OR '.join(generic_anchors)})"

# # 5. Treder Library
# Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

# # 6. Text-Suche: Core-Begriffe und verankerte breite Begriffe
# grg_core_terms = [
#     '(general AND relativi*)', '(allgemein* AND relativi*)',
#     '"relativité générale"', '"relatività generale"',
#     '"Gravité quantique"', '"Gravità quantistica"', 'Quantengravitation', '"quantum gravity"',
#     '(einheitlich* AND feld*)', '"champ unifié"', '(unified AND field*)'
# ]

# broad_terms = ['gravit*', 'cosmo*', 'Kosmo*']
# anchored_broad = f"({' OR '.join(broad_terms)}) AND ({' OR '.join(generic_anchors)})"

# Set_E = f"abs:({' OR '.join(grg_core_terms)} OR ({anchored_broad})) AND year:1911-2000"

# # Finale Query
# QUERY = f"({Set_A}) OR ({Set_B}) OR ({Set_C}) OR ({Set_T}) OR ({Set_E})"

### 1.2 Execute Search

Run the ADS search and persist bibcodes/results to this run folder.
This creates a reproducible input snapshot for later phases.


In [6]:
if START_AT_PHASE <= 1:
    from ads_bib.search import search_ads

    bibcodes, references, esources, fulltext_urls = search_ads(
        QUERY, ADS_TOKEN, raw_dir=paths["raw"], force_refresh=FORCE_REFRESH,
    )
else:
    logger.info("Skipping Phase 1 Search (START_AT_PHASE=%s)", START_AT_PHASE)

Starting ADS search ...
  382 records fetched ...
Done. Bibcodes: 382 | Unique refs: 499 | Total refs: 796
Saved: search_results_20260227_165757.pkl
Latest: search_results_latest.pkl


### 1.3 Export & Resolve Metadata

Resolve publications and references to structured metadata tables.
This normalizes schemas before translation/tokenization/topic modeling.


In [7]:
if START_AT_PHASE <= 1:
    from ads_bib.export import resolve_dataset

    publications, refs = resolve_dataset(
        bibcodes, references, esources, fulltext_urls, ADS_TOKEN,
        cache_dir=paths["raw"], force_refresh=FORCE_REFRESH,
    )
else:
    logger.info("Skipping Phase 1 Export (START_AT_PHASE=%s)", START_AT_PHASE)

=== Exporting publications ===
Exporting 382 bibcodes in 1 chunks (5 workers) ...


Exporting:   0%|          | 0/382 [00:00<?, ?it/s]

Publications: 382 records
=== Exporting references (499 unique) ===
Exporting 499 bibcodes in 1 chunks (5 workers) ...


Exporting:   0%|          | 0/499 [00:00<?, ?it/s]

References: 499 records


In [8]:
if START_AT_PHASE <= 1:
    preview_cols = [
        c for c in ("Bibcode", "Year", "Author", "Title", "References")
        if c in publications.columns
    ]
    logger.info(
        "Phase 1 preview: publications=%s rows, columns=%s",
        f"{len(publications):,}",
        len(publications.columns),
    )
    if preview_cols:
        display(publications[preview_cols].head(10))
    else:
        display(publications.head(10))

Phase 1 preview: publications=382 rows, columns=18


,Bibcode,Year,Author,Title,References
0,1971grun.book.....T,1971,"[Treder, H.J.]",Gravitationstheorie und Äquivalenzprinzip.,[]
1,1983AN....304..145T,1983,"[Treder, H.J.]",The problem of the Grand Unification Theory,[1950sts..book.....S]
2,1957AnP...454..369T,1957,"[Treder, H.]",Stromladungsdefinition und elektrische Kraft i...,[1953PhRv...92.1567C]
3,1967AnP...475..194T,1967,"[Treder, HansJürgen]",Das makroskopische Gravitationsfeld in der ein...,"[1916AnP...354..769E, 1940PhRv...57..147R, 195..."
4,1972drdt.book.....T,1972,"[Treder, H.J.]",Die Relativität der Trägheit.,[]
5,1997GReGr..29..455V,1997,"[von Borzeszkowski, HorstHeino, Treder, HansJü...",The Weyl-Cartan Space Problem in Purely Affine...,"[1950sts..book.....S, 1971GReGr...2..313T, 199..."
6,1981AN....302..275T,1981,"[Treder, H. J., Jackisch, G.]",On Soldners Value of Newtonian Deflection of L...,"[1970AnP...480..315T, 1974AnP...486..325T]"
7,1975AnP...487..383T,1975,"[Treder, H.J.]",Zur unitarisierten Gravitationstheorie mit lan...,"[1963ForPh..11...81T, 1973AnP...485..229T, 197..."
8,1988mqg..book.....V,1988,"[von Borzeszkowski, H.H., Treder, H.J.]",The meaning of quantum gravity.,[]
9,1980AnP...492..250T,1980,"[Treder, H.J.]",Einstein's Hermitian theory of relativity as u...,"[1957AnP...454..369T, 1975AnP...487..375T]"


---
# Phase 2: Translation

Detect non-English titles and abstracts with fasttext,
then translate them using either OpenRouter (API) or a local HuggingFace model.

### 2.1 Translation Configuration

Choose provider/model and translation target language.
Keep settings explicit so costs and outputs are reproducible.


In [9]:
# ── CONFIGURE ─────────────────────────────────────────
# Provider: 'openrouter' (API; requires OPENROUTER_API_KEY) or 'huggingface' (local)
TRANSLATION_PROVIDER = "gguf"
TRANSLATION_MODEL = "mradermacher/translategemma-4b-it-GGUF"
TRANSLATION_API_KEY = os.getenv("OPENROUTER_API_KEY")
TRANSLATION_MAX_WORKERS = 10
TRANSLATION_MAX_TOKENS = 2048

# Path to fasttext language ID model:
# https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
FASTTEXT_MODEL = paths["models"] / "lid.176.bin"
# ──────────────────────────────────────────────────────

### 2.2 Language Detection

Detect language tags for configured text columns.
Only non-target-language rows are translated in the next step.


In [10]:
if START_AT_PHASE <= 2:
    from ads_bib.translate import detect_languages
    
    logger.info("=== Publications ===")
    publications = detect_languages(publications, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)
    
    logger.info("\n=== References ===")
    refs = detect_languages(refs, ["Title", "Abstract"], model_path=FASTTEXT_MODEL)
else:
    logger.info("Skipping Phase 2 Translation (START_AT_PHASE=%s)", START_AT_PHASE)

=== Publications ===
  Title: 183 non-English entries detected
  Abstract: 35 non-English entries detected

=== References ===
  Title: 185 non-English entries detected
  Abstract: 36 non-English entries detected


### 2.3 Translate Non-English Entries

Translate non-English rows and track token/cost metadata.
This harmonizes text fields for downstream NLP and topic modeling.


In [11]:
if START_AT_PHASE <= 2:
    from ads_bib.translate import translate_dataframe

    logger.info("=== Translating Publications ===")
    publications, cost_pubs = translate_dataframe(
        publications, ["Title", "Abstract"],
        provider=TRANSLATION_PROVIDER,
        model=TRANSLATION_MODEL,
        api_key=TRANSLATION_API_KEY,
        max_workers=TRANSLATION_MAX_WORKERS,
        max_translation_tokens=TRANSLATION_MAX_TOKENS,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )

    logger.info("\n=== Translating References ===")
    refs, cost_refs = translate_dataframe(
        refs, ["Title", "Abstract"],
        provider=TRANSLATION_PROVIDER,
        model=TRANSLATION_MODEL,
        api_key=TRANSLATION_API_KEY,
        max_workers=TRANSLATION_MAX_WORKERS,
        max_translation_tokens=TRANSLATION_MAX_TOKENS,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )
else:
    logger.info("Skipping Phase 2 Translation (START_AT_PHASE=%s)", START_AT_PHASE)

=== Translating Publications ===


translategemma-4b-it.Q4_K_M.gguf:   0%|          | 0.00/2.49G [00:00<?, ?B/s]

c:\Users\rapha\anaconda3\envs\ADS_env\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
  Title: translating 183 entries with gguf/mradermacher/translategemma-4b-it-GGUF ...


  Title:   0%|          | 0/183 [00:00<?, ?it/s]

Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4

...

Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4

  Title:   0%|          | 0/185 [00:00<?, ?it/s]

Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4

  Abstract:   0%|          | 0/36 [00:00<?, ?it/s]

Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4_K_M.gguf
Loading GGUF translation model: C:\Users\rapha\.cache\huggingface\hub\models--mradermacher--translategemma-4b-it-GGUF\snapshots\35a7486e128b19642cdc72d7b91b21ba388aaf42\translategemma-4b-it.Q4

### 2.4 Save Translation Checkpoint

Persist translated data to global cache and run folder for Phase 3 restarts.

In [12]:
if START_AT_PHASE <= 2:
    from ads_bib._utils.checkpoints import save_translated_checkpoint

    save_translated_checkpoint(
        publications,
        refs,
        cache_dir=paths["cache"],
        run_data_dir=run.paths["data"],
    )

Translated checkpoint saved to global cache and local run folder.


---
# Phase 3: Tokenization

Create `full_text` (Title + Abstract) and tokenize publications with spaCy.
References are intentionally skipped in this phase for runtime stability.

In [13]:
import os

# ── CONFIGURE ─────────────────────────────────────────
TOKENIZATION_SPACY_MODEL = "en_core_web_md"
TOKENIZATION_BATCH_SIZE = 512
TOKENIZATION_N_PROCESS = min(max((os.cpu_count() or 1) - 1, 1), 8)
TOKENIZATION_DISABLE = ("ner", "parser", "textcat")
# ──────────────────────────────────────────────────────

if START_AT_PHASE <= 3:
    from ads_bib._utils.checkpoints import load_translated_checkpoint
    from ads_bib.tokenize import ensure_spacy_model, tokenize_texts

    if START_AT_PHASE == 3:
        logger.info("Loading translated data from global cache (START_AT_PHASE=3) ...")
        publications, refs = load_translated_checkpoint(
            cache_dir=paths["cache"],
            run_data_dir=run.paths["data"],
        )

    model_to_use, preloaded_nlp = ensure_spacy_model(
        spacy_model=TOKENIZATION_SPACY_MODEL,
        fallback_model="en_core_web_lg",
        spacy_disable=TOKENIZATION_DISABLE,
        auto_download=True,
    )

    logger.info("Tokenizing publications only ...")
    publications = tokenize_texts(
        publications,
        spacy_model=model_to_use,
        nlp=preloaded_nlp,
        batch_size=TOKENIZATION_BATCH_SIZE,
        n_process=TOKENIZATION_N_PROCESS,
        spacy_disable=TOKENIZATION_DISABLE,
    )
    logger.info("refs tokenization skipped by design.")
else:
    logger.info("Skipping Phase 3 Tokenization (START_AT_PHASE=%s)", START_AT_PHASE)

Tokenizing publications only ...
Tokenizing 382 documents with en_core_web_md (n_process=8, batch_size=512) ...


Tokenization:   0%|          | 0/382 [00:00<?, ?it/s]

  Done.
refs tokenization skipped by design.


### Checkpoint: Save Phase 3 Results

Persist tokenized publications and translated references for Phase 4 restarts.
This avoids rerunning API-heavy earlier phases.


In [14]:
if START_AT_PHASE <= 3:
    from ads_bib._utils.checkpoints import save_phase3_checkpoint

    save_phase3_checkpoint(
        publications,
        refs,
        cache_dir=paths["cache"],
        run_data_dir=run.paths["data"],
    )


Phase 3 checkpoint saved (publications tokenized; refs retained without tokenization).


---
# Phase 4: Author Name Disambiguation (Placeholder)

This step will use an external AND package once it's ready.
For now, the pipeline continues without disambiguation.

In [15]:
# === AND PLACEHOLDER ===
# Uncomment when the AND package is available:
#
# from and_package import disambiguate
# publications = disambiguate(publications)
# refs = disambiguate(refs)

logger.info("AND step skipped (placeholder). Continuing without disambiguation.")

AND step skipped (placeholder). Continuing without disambiguation.


---
# Phase 5: Topic Modeling & Curation

Use BERTopic to discover thematic clusters, visualize with datamapplot,
then remove unwanted clusters to curate the dataset.

**Important:** Set parameters below based on your dataset size from Phase 1.

### 5.1 Embedding Configuration

Configure embedding provider/model and optional sampling.
These settings strongly affect topic quality and runtime/cost.


In [16]:
# ── CONFIGURE ─────────────────────────────────────────
# Providers: 'openrouter' (API), 'huggingface_api' (remote via LiteLLM), 'local' (SentenceTransformers)
EMBEDDING_PROVIDER = "local"
EMBEDDING_MODEL = "google/embeddinggemma-300m"
EMBEDDING_API_KEY = os.getenv("OPENROUTER_API_KEY")
EMBEDDING_MAX_WORKERS = 10

# For testing: set SAMPLE_SIZE to an integer (e.g. 200). None = full dataset.
SAMPLE_SIZE = None
# ──────────────────────────────────────────────────────

### 5.2 Compute Embeddings

Create or load cached embeddings for `full_text`.
Caching keeps repeated runs fast and reproducible.


In [17]:
if START_AT_PHASE >= 4:
    from ads_bib._utils.checkpoints import load_phase3_checkpoint

    publications, refs = load_phase3_checkpoint(
        cache_dir=paths["cache"], run_data_dir=run.paths["data"],
    )

In [18]:
if START_AT_PHASE <= 5:
    from ads_bib.topic_model import compute_embeddings

    df = publications.copy()
    if SAMPLE_SIZE is not None:
        df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
        logger.info("SAMPLING: %s documents", f"{len(df):,}")

    documents = df["full_text"].tolist()

    embeddings = compute_embeddings(
        documents,
        provider=EMBEDDING_PROVIDER,
        model=EMBEDDING_MODEL,
        cache_dir=paths["embeddings_cache"],
        max_workers=EMBEDDING_MAX_WORKERS,
        api_key=EMBEDDING_API_KEY,
        openrouter_cost_mode=OPENROUTER_COST_MODE,
        cost_tracker=tracker,
    )
else:
    logger.info("Skipping Phase 5.2 Compute Embeddings (START_AT_PHASE=%s)", START_AT_PHASE)

  Computing embeddings with local/google/embeddinggemma-300m ...
TensorFlow version 2.8.4 available.
  Loading local model: google/embeddinggemma-300m
Use pytorch device_name: cpu
Load pretrained SentenceTransformer: google/embeddinggemma-300m


modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

c:\Users\rapha\anaconda3\envs\ADS_env\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rapha\.cache\huggingface\hub\models--google--embeddinggemma-300m. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

14 prompts are loaded, with the keys: ['query', 'document', 'BitextMining', 'Clustering', 'Classification', 'InstructionRetrieval', 'MultilabelClassification', 'PairClassification', 'Reranking', 'Retrieval', 'Retrieval-query', 'Retrieval-document', 'STS', 'Summarization']


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

  Saved: embeddings_local_google_embeddinggemma-300m.npz
  Embeddings: (382, 768)


### 5.3 Dimensionality Reduction Configuration

- Methods: `pacmap` (fast) or `umap` (preferred for hierarchical Toponymy structure).
- Heuristic: smaller/sparser sets use `n_neighbors` ~15; larger/denser sets often benefit from 50-60.
- Use `min_dist=0.0` for 5D clustering vectors and `min_dist=0.1` for 2D visualization.


In [19]:
# ── CONFIGURE ─────────────────────────────────────────
DIM_REDUCTION_METHOD = "pacmap"  # PaCMAP als lokaler/globaler Kompromiss für großes GRG-Korpus

PARAMS_5D = dict(n_neighbors=60, metric="angular", random_state=RANDOM_SEED)
PARAMS_2D = dict(n_neighbors=60, metric="angular", random_state=RANDOM_SEED)
# ──────────────────────────────────────────────────────

In [20]:
if START_AT_PHASE <= 5:
    from ads_bib.topic_model import reduce_dimensions

    reduced_5d, reduced_2d = reduce_dimensions(
        embeddings,
        method=DIM_REDUCTION_METHOD,
        params_5d=PARAMS_5D,
        params_2d=PARAMS_2D,
        random_state=RANDOM_SEED,
        cache_dir=paths["dim_reduction_cache"],
        embedding_id=f"{EMBEDDING_PROVIDER}/{EMBEDDING_MODEL}",
    )
else:
    logger.info("Skipping Phase 5.3 Dimensionality Reduction (START_AT_PHASE=%s)", START_AT_PHASE)

Reduction (PACMAP):   0%|          | 0/2 [00:00<?, ?it/s]

  Computing 5d_local_google_embeddinggemma-300m_pacmap_nn60_minddef_metricangular_rs42 with PACMAP ...
Note: `n_components != 2` have not been thoroughly tested.
  Saved: reduced_5d_local_google_embeddinggemma-300m_pacmap_nn60_minddef_metricangular_rs42.npz
  Computing 2d_local_google_embeddinggemma-300m_pacmap_nn60_minddef_metricangular_rs42 with PACMAP ...
  Saved: reduced_2d_local_google_embeddinggemma-300m_pacmap_nn60_minddef_metricangular_rs42.npz
  5D shape: (382, 5), 2D shape: (382, 2)


### 5.4 Clustering Configuration

**Adjust clustering granularity based on dataset size.**
- `MIN_CLUSTER_SIZE` controls BERTopic/HDBSCAN topic granularity (roughly ~0.1% of docs).
- `BASE_MIN_CLUSTER_SIZE` controls Toponymy micro-cluster granularity.
- `TOPONYMY_LAYER_INDEX` selects the fallback primary layer for visualization.


In [21]:
# ── CONFIGURE ─────────────────────────────────────────
CLUSTERING_METHOD = "fast_hdbscan"  # Schnellste Variante für 87k

n_docs = len(df)
MIN_CLUSTER_SIZE = max(15, int(n_docs * 0.001)) # ca. 87 bei 87k Dokumenten
BASE_MIN_CLUSTER_SIZE = max(55, int(n_docs * 0.0007))
# ──────────────────────────────────────────────────────

logger.info("Dataset: %s documents", f"{n_docs:,}")
logger.info("  MIN_CLUSTER_SIZE=%s", MIN_CLUSTER_SIZE)
logger.info("  BASE_MIN_CLUSTER_SIZE=%s", BASE_MIN_CLUSTER_SIZE)

CLUSTER_PARAMS = dict(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=2, # Reduziert Outlier drastisch, da weniger Dichte für Cluster-Zugehörigkeit gefordert wird
    cluster_selection_method="eom",
    cluster_selection_epsilon=0.05, # Absorbiert Randpunkte in bestehende Cluster
)

TOPONYMY_CLUSTER_PARAMS = dict(
    min_clusters=10,
    min_samples=2,
    base_min_cluster_size=BASE_MIN_CLUSTER_SIZE,
)

# EVoC uses a backend-specific constructor in some versions.
# Keep this dict conservative; unsupported keys are filtered in fit_toponymy.
TOPONYMY_EVOC_CLUSTER_PARAMS = dict(
    min_clusters=10,
    min_samples=2,
    base_min_cluster_size=BASE_MIN_CLUSTER_SIZE,
    noise_level=0.35,
    n_neighbors=15,
    n_epochs=35,
)

TOPONYMY_LAYER_INDEX = 1  # Broader Toponymy layer to reduce -1 outliers on large corpora.

Dataset: 382 documents
  MIN_CLUSTER_SIZE=15
  BASE_MIN_CLUSTER_SIZE=55


### 5.5 Backend & LLM Configuration

Backend matrix:
- `bertopic`: BERTopic on 5D reduced vectors + optional outlier reassignment refresh.
- `toponymy`: Toponymy + `ToponymyClusterer` on 5D reduced vectors.
- `toponymy_evoc`: Toponymy + `EVoCClusterer` on raw embeddings (no 5D clustering vectors).

LLM provider matrix:
- BERTopic: `local`, `huggingface_api`, `openrouter`
- Toponymy / Toponymy+EVoC: `local` (Toponymy HuggingFaceNamer) or `openrouter`

`MIN_DF` guidance:
- small sample runs: often `1`
- full runs: usually `2+`


In [33]:
# ── CONFIGURE ─────────────────────────────────────────
TOPIC_BACKEND = "bertopic"  # 'bertopic', 'toponymy', or 'toponymy_evoc'

# BERTopic providers: 'local', 'huggingface_api', 'openrouter'
# Toponymy providers: 'local' (HuggingFaceNamer) or 'openrouter'
LLM_PROVIDER = "local"
LLM_MODEL = "google/gemma-3-1b-it"
# Quality alternative (slower/heavier local labeling): "google/gemma-3-4b-it"
LLM_API_KEY = os.getenv("OPENROUTER_API_KEY")
BERTOPIC_LABEL_MAX_TOKENS = 128
TOPONYMY_LOCAL_LABEL_MAX_TOKENS = 256

PIPELINE_MODELS = ["POS", "KeyBERT", "MMR"]
PARALLEL_MODELS = ["MMR", "POS", "KeyBERT"]

TOPONYMY_EMBEDDING_MODEL = EMBEDDING_MODEL
TOPONYMY_MAX_WORKERS = 10

# MIN_DF=5 filtert extrem seltene Wörter heraus. Beschleunigt c-TF-IDF bei 87k enorm und spart RAM.
MIN_DF = 5
# ──────────────────────────────────────────────────────

### 5.5b Fit Topic Model

Run the selected backend (BERTopic or Toponymy) on reduced vectors.
This is the most compute/cost-intensive step of the pipeline.

In [34]:
if START_AT_PHASE <= 5:
    import numpy as np
    from ads_bib.topic_model import fit_bertopic, fit_toponymy, reduce_outliers

    if TOPIC_BACKEND == "bertopic":
        topic_model = fit_bertopic(
            documents,
            reduced_5d,
            llm_provider=LLM_PROVIDER,
            llm_model=LLM_MODEL,
            llm_max_new_tokens=BERTOPIC_LABEL_MAX_TOKENS,
            pipeline_models=PIPELINE_MODELS,
            parallel_models=PARALLEL_MODELS,
            min_df=MIN_DF,
            clustering_method=CLUSTERING_METHOD,
            clustering_params=CLUSTER_PARAMS,
            api_key=LLM_API_KEY,
            openrouter_cost_mode=OPENROUTER_COST_MODE,
            cost_tracker=tracker,
        )

        topics = np.array(topic_model.topics_)
        topics = reduce_outliers(
            topic_model,
            documents,
            topics,
            reduced_5d,
            threshold=0.4,
            llm_provider=LLM_PROVIDER,
            llm_model=LLM_MODEL,
            api_key=LLM_API_KEY,
            openrouter_cost_mode=OPENROUTER_COST_MODE,
            cost_tracker=tracker,
        )
        topic_info = topic_model.get_topic_info()

    elif TOPIC_BACKEND in {"toponymy", "toponymy_evoc"}:
        _cluster_params = (
            TOPONYMY_EVOC_CLUSTER_PARAMS if TOPIC_BACKEND == "toponymy_evoc"
            else TOPONYMY_CLUSTER_PARAMS
        )
        topic_model, topics, topic_info = fit_toponymy(
            documents,
            embeddings,
            reduced_5d,
            backend=TOPIC_BACKEND,
            layer_index=TOPONYMY_LAYER_INDEX,
            llm_provider=LLM_PROVIDER,
            llm_model=LLM_MODEL,
            embedding_model=TOPONYMY_EMBEDDING_MODEL,
            local_llm_max_new_tokens=TOPONYMY_LOCAL_LABEL_MAX_TOKENS,
            api_key=LLM_API_KEY,
            openrouter_cost_mode=OPENROUTER_COST_MODE,
            max_workers=TOPONYMY_MAX_WORKERS,
            clusterer_params=_cluster_params,
            cost_tracker=tracker,
        )

    else:
        raise ValueError(f"Invalid TOPIC_BACKEND '{TOPIC_BACKEND}'.")

    tracker.log_steps_summary([
        "llm_labeling", "llm_labeling_post_outliers",
        "llm_labeling_toponymy", "llm_labeling_toponymy_evoc",
        "toponymy_embeddings",
    ])
else:
    logger.info("Skipping Phase 5.5b Fit Topic Model (START_AT_PHASE=%s)", START_AT_PHASE)

Preparing BERTopic components (pipeline=['POS', 'KeyBERT', 'MMR'], parallel=['MMR', 'POS', 'KeyBERT']) ...
  Initializing POS keyword extraction model: en_core_web_sm
  Loading local LLM: google/gemma-3-1b-it


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Fitting BERTopic (LLM: local/google/gemma-3-1b-it) ...


BERTopic fit:   0%|          | 0/1 [00:00<?, ?it/s]

2026-02-27 18:14:45,202 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-27 18:14:45,203 - BERTopic - Dimensionality - Completed ✓
2026-02-27 18:14:45,205 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-27 18:14:45,217 - BERTopic - Cluster - Completed ✓
2026-02-27 18:14:45,224 - BERTopic - Representation - Fine-tuning topics using representation models.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_toke

BERTopic refresh:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

### 5.5c Build Topic DataFrame

Merge topic assignments, 2D coordinates, and embeddings into the main DataFrame.

In [35]:
if START_AT_PHASE <= 5:
    from ads_bib.topic_model import build_topic_dataframe

    df = build_topic_dataframe(
        df,
        topic_model,
        topics,
        reduced_2d,
        embeddings,
        topic_info=topic_info,
    )
    display(topic_info)
else:
    logger.info("Skipping Phase 5.5c Build Topic DataFrame (START_AT_PHASE=%s)", START_AT_PHASE)

,Topic,Count,Name,Representation,MMR,POS,KeyBERT,Representative_Docs
0,-1,16,-1_\ntopic: Gravitation\ntopic: Einsteins Theo...,[\ntopic: Gravitation\ntopic: Einsteins Theori...,"[des, gravitation, theories, die, eddington, e...","[und, gravitation, theories, theory, der, , , ...","[gravitation, einstein, theory, theories, zur,...",[Die Lichtgeschwindigkeit im Gravitationsfeld ...
1,0,144,0_\ntopic: Gravitational field equations\ntopi...,[\ntopic: Gravitational field equations\ntopic...,"[einstein, die, theory, gravitation, und, theo...","[der, die, theory, gravitation, und, theories,...","[gravitation, einstein, theories, theory, eddi...",[Die post-Newtonschen Effekte in der Gravodyna...
2,1,118,1_\n```python\ndef generate_topic_label(docume...,[\n```python\ndef generate_topic_label(documen...,"[der, die, von, gravitation, des, einstein, th...","[zu, gravitation, theory, theories, , , , , , ]","[einstein, gravitation, und, der, theory, des,...","[Friedrich Engels und die Geschichte der ""blac..."
3,2,60,2_\n```python\ndef generate_topic_label(docume...,[\n```python\ndef generate_topic_label(documen...,"[des, die, eddington, einstein, gravitation, v...","[und, gravitation, zu, problem, theory, theori...","[und, zu, des, zur, der, theory, gravitation, ...",[Kants Kosmologie und der physische Teil des n...
4,3,44,3_\n```python\ntopic: Gravitation\n```\n```\nt...,[\n```python\ntopic: Gravitation\n```\n```\nto...,"[problem, theory, gravitation, einstein, und, ...","[problem, theory, gravitation, der, und, theor...","[gravitation, einstein, theory, theories, eddi...",[Singularities in the General Theory of Relati...


### 5.6 Visualize Topics

Render an interactive topic map from 2D coordinates and topic labels.
Use this view to inspect cluster semantics before curation.


In [36]:
if START_AT_PHASE <= 5:
    from ads_bib.visualize import create_topic_map

    plot = create_topic_map(
        df,
        title="ADS Bibliometric Map",
        subtitle=f"Topics labeled with {LLM_PROVIDER}/{LLM_MODEL}",
        dark_mode=True,
        output_path=run.paths["plots"] / "topic_map.html",
    )
    # plot  # uncomment to display inline
else:
    logger.info("Skipping Phase 5.6 Visualize Topics (START_AT_PHASE=%s)", START_AT_PHASE)

Saved: topic_map.html


### 5.7 Curate Dataset

Review the cluster summary, then remove clusters that don't fit your research question.

In [37]:
if START_AT_PHASE <= 5:
    from ads_bib.curate import get_cluster_summary, remove_clusters

    display(get_cluster_summary(df))
else:
    logger.info("Skipping Phase 5.7 Curate Dataset (START_AT_PHASE=%s)", START_AT_PHASE)

,topic_id,Count,Percentage,Label
1,0,144,37.7,0_\ntopic: Gravitational field equations\ntopi...
2,1,118,30.9,1_\n```python\ndef generate_topic_label(docume...
3,2,60,15.7,2_\n```python\ndef generate_topic_label(docume...
4,3,44,11.5,3_\n```python\ntopic: Gravitation\n```\n```\nt...
0,-1,16,4.2,-1_\ntopic: Gravitation\ntopic: Einsteins Theo...


### Checkpoint: Save Curated Dataset

Save the curated topic dataset as the handoff for citation analysis.
This provides a stable artifact for Phase 6 and external reuse.


In [38]:
if START_AT_PHASE <= 5:
    from ads_bib._utils.io import save_parquet

    # Ensure References is list type for Parquet
    df["References"] = df["References"].apply(lambda x: x if isinstance(x, list) else [])

    save_parquet(df, run.paths["data"] / "curated_dataset.parquet")
    logger.info("Curated dataset: %s documents", f"{len(df):,}")
else:
    logger.info("Skipping Checkpoint: Save Curated Dataset (START_AT_PHASE=%s)", START_AT_PHASE)

Curated dataset: 382 documents


In [39]:
if START_AT_PHASE <= 5:
    from ads_bib._utils.io import load_parquet

    df = load_parquet(run.paths["data"] / "curated_dataset.parquet")
    display(df.head(10))
else:
    logger.info("Skipping Checkpoint: Load Curated Dataset (START_AT_PHASE=%s)", START_AT_PHASE)

,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,...,full_text,tokens,embedding_2d_x,embedding_2d_y,topic_id,Name,MMR,POS,KeyBERT,full_embeddings
0,1971grun.book.....T,"[Treder, H.J.]",Gravitationstheorie und Äquivalenzprinzip.,1971,Gravitationstheorie und Äquivalenzprinzip,grun.book,,,,,...,Gravitationstheorie und Äquivalenzprinzip..,"[gravitationstheorie, und, äquivalenzprinzip]",2.083590,-1.009248,1,1_\n```python\ndef generate_topic_label(docume...,"der, die, von, gravitation, des, einstein, the...","zu, gravitation, theory, theories, , , , , ,","einstein, gravitation, und, der, theory, des, ...","[-0.007528835, 0.013286961, -0.012361329, 0.03..."
1,1983AN....304..145T,"[Treder, H.J.]",The problem of the Grand Unification Theory,1983,Astronomische Nachrichten,AN,4,304,145,151,...,The problem of the Grand Unification Theory. T...,"[problem, grand, unification, theory, evolutio...",-1.894265,0.147704,0,0_\ntopic: Gravitational field equations\ntopi...,"einstein, die, theory, gravitation, und, theor...","der, die, theory, gravitation, und, theories, ...","gravitation, einstein, theories, theory, eddin...","[0.009722454, 0.0146043375, 0.02066006, 0.0162..."
2,1957AnP...454..369T,"[Treder, H.]",Stromladungsdefinition und elektrische Kraft i...,1957,Annalen der Physik,AnP,6,454,369,380,...,Stromladungsdefinition und elektrische Kraft i...,"[stromladungsdefinition, und, elektrische, kra...",-1.782198,-0.804118,0,0_\ntopic: Gravitational field equations\ntopi...,"einstein, die, theory, gravitation, und, theor...","der, die, theory, gravitation, und, theories, ...","gravitation, einstein, theories, theory, eddin...","[-0.015809545, 0.035810255, -0.006606995, 0.03..."
3,1967AnP...475..194T,"[Treder, HansJürgen]",Das makroskopische Gravitationsfeld in der ein...,1967,Annalen der Physik,AnP,3,475,194,206,...,Das makroskopische Gravitationsfeld in der ein...,"[das, makroskopische, gravitationsfeld, der, e...",-2.573581,-0.781189,0,0_\ntopic: Gravitational field equations\ntopi...,"einstein, die, theory, gravitation, und, theor...","der, die, theory, gravitation, und, theories, ...","gravitation, einstein, theories, theory, eddin...","[-0.011236075, 0.040593218, -0.03224795, 0.003..."
4,1972drdt.book.....T,"[Treder, H.J.]",Die Relativität der Trägheit.,1972,Die Relativität der Trägheit,drdt.book,,,,,...,Die Relativität der Trägheit..,"[die, relativität, der, trägheit]",1.580791,-0.416789,1,1_\n```python\ndef generate_topic_label(docume...,"der, die, von, gravitation, des, einstein, the...","zu, gravitation, theory, theories, , , , , ,","einstein, gravitation, und, der, theory, des, ...","[-0.09778604, -0.048137885, 0.030424133, -0.03..."
5,1997GReGr..29..455V,"[von Borzeszkowski, HorstHeino, Treder, HansJü...",The Weyl-Cartan Space Problem in Purely Affine...,1997,General Relativity and Gravitation,GReGr,4,29,455,466,...,The Weyl-Cartan Space Problem in Purely Affine...,"[weyl, cartan, space, problem, purely, affine,...",-2.397137,-1.219562,0,0_\ntopic: Gravitational field equations\ntopi...,"einstein, die, theory, gravitation, und, theor...","der, die, theory, gravitation, und, theories, ...","gravitation, einstein, theories, theory, eddin...","[-0.05071241, 0.041779127, 0.0068002217, 0.021..."
6,1981AN....302..275T,"[Treder, H. J., Jackisch, G.]",On Soldners Value of Newtonian Deflection of L...,1981,Astronomische Nachrichten,AN,6,302,275,,...,On Soldners Value of Newtonian Deflection of L...,"[soldners, value, newtonian, deflection, light]",1.462896,-0.078913,1,1_\n```python\ndef generate_topic_label(docume...,"der, die, von, gravitation, des, einstein, the...","zu, gravitation, theory, theories, , , , , ,","einstein, gravitation, und, der, theory, des, ...","[-0.090161584, 0.049392313, 0.003038768, 0.015..."
7,1975AnP...487..383T,"[Treder, H.J.]",Zur unitarisierten Gravitationstheorie mit lan...,1975,Annalen der Physik,AnP,,487,383,400,...,Zur unitarisierten Gravitationstheorie mit lan...,"[zu

---
# Phase 6: Citation Networks

Compute citation networks from the curated dataset and export
to GEXF (Gephi), Graphology JSON (Sigma.js), CSV, and/or Web of Science format.

### 6.1 Citation Configuration

Set network metrics, thresholds, filters, and export formats.
These parameters define which citation structures are constructed.

In [40]:
# ── CONFIGURE ─────────────────────────────────────────
CITE_METRICS = ["direct", "co_citation", "bibliographic_coupling", "author_co_citation"]
MIN_COUNTS = {
    "direct": 1,
    "co_citation": 1,
    "bibliographic_coupling": 1,
    "author_co_citation": 1,
}
AUTHORS_FILTER = None
OUTPUT_FORMAT = "gexf"  # 'gexf', 'graphology', 'csv', 'all'
CLUSTERS_TO_REMOVE = []
# ──────────────────────────────────────────────────────

# Snapshot the full configuration once all phase configs are defined.
run.save_config(globals())

Snapshot of configuration saved to config_used.yaml (47 parameters tracked).


### 6.2 Build Node Table & Compute Networks

Build node metadata and compute selected citation networks.
Outputs are exported for Gephi/Sigma/CSV workflows.


In [41]:
if START_AT_PHASE <= 6:
    if START_AT_PHASE == 6:
        from ads_bib._utils.checkpoints import load_phase3_checkpoint

        publications, refs = load_phase3_checkpoint(
            cache_dir=paths["cache"], run_data_dir=run.paths["data"],
        )

    from ads_bib.citations import (
        build_all_nodes,
        build_citation_inputs_from_publications,
        export_wos_format,
        process_all_citations,
    )

    bibcodes, references = build_citation_inputs_from_publications(publications)
    all_nodes = build_all_nodes(publications, refs)

    results = process_all_citations(
        bibcodes=bibcodes,
        references=references,
        publications=publications,
        ref_df=refs,
        all_nodes=all_nodes,
        metrics=CITE_METRICS,
        min_counts=MIN_COUNTS,
        authors_filter=AUTHORS_FILTER,
        output_format=OUTPUT_FORMAT,
        output_dir=run.paths["data"],
    )
else:
    logger.info("Skipping Phase 6 Citation Networks (START_AT_PHASE=%s)", START_AT_PHASE)

All nodes: 773


Direct citation:            0/2 [00:00]

Direct citations:   0%|          | 0/382 [00:00<?, ?it/s]

GEXF nodes:   0%|          | 0/597 [00:00<?, ?it/s]

GEXF edges:   0%|          | 0/796 [00:00<?, ?it/s]

...

Author co-citation:            0/2 [00:00]

Author co-citation detail:   0%|          | 0/127 [00:00<?, ?it/s]

GEXF nodes:   0%|          | 0/127 [00:00<?, ?it/s]

GEXF edges:   0%|          | 0/2079 [00:00<?, ?it/s]

  Author co-citation — 127 nodes, 2,079 edges, no filter, 758.6 KB


### 6.3 Export Web of Science Format

Export the curated corpus in WOS text format.
This supports downstream tooling such as CiteSpace/VOSviewer.


In [42]:
suffix = "_filtered" if AUTHORS_FILTER else ""
export_wos_format(
    publications, refs,
    output_path=run.paths["data"] / f"download_wos_export{suffix}.txt",
)

  WOS format: download_wos_export.txt


---
# Summary

Final dataset statistics and output file listing.

In [43]:
logger.info("=" * 60)
logger.info("PIPELINE COMPLETE")
logger.info("=" * 60)
logger.info("Publications:     %s", f"{len(publications):,}")
logger.info("References:       %s", f"{len(refs):,}")
logger.info("Curated dataset:  %s", f"{len(df):,}")
logger.info("Topics found:     %s", df["topic_id"].nunique())
logger.info("")
logger.info("Output files:")
for root, dirs, files in os.walk(run.paths["root"]):
    for f in sorted(files):
        fpath = Path(root) / f
        size_mb = fpath.stat().st_size / 1024 / 1024
        logger.info("  %s (%.1f MB)", fpath.relative_to(run.paths["root"]), size_mb)

logger.info("")
logger.info(tracker.compact_summary())

PIPELINE COMPLETE
Publications:     382
References:       499
Curated dataset:  382
Topics found:     5

Output files:
  config_used.yaml (0.0 MB)
  data\author_co_citation.gexf (0.7 MB)
  data\bibliographic_coupling.gexf (0.8 MB)
  data\co_citation.gexf (1.9 MB)
  data\curated_dataset.parquet (2.0 MB)
  data\direct.gexf (1.4 MB)
  data\download_wos_export.txt (0.3 MB)
  data\publications_translated.json (0.5 MB)
  data\publications_translated_tokenized.json (0.7 MB)
  data\references_translated.json (0.7 MB)
  plots\topic_map.html (0.3 MB)

CostTracker: no entries
